In [1]:
!pip install kubernetes jinja2 pyyaml --quiet

In [2]:
from dotenv import load_dotenv
import os

local_base_dir = os.path.expanduser("~/shared")
params_file = f"{local_base_dir}/output-3.env"
load_dotenv(params_file)

model_path = os.environ.get('MODEL_PATH') #"meta-llama/Llama-3.2-1B-Instruct-tuned"
model_name = os.environ.get('MODEL_NAME') #"meta-llama/Llama"
version = os.environ.get('MODEL_VERSION') #"3.2-1B-Instruct-tuned-4"

In [3]:
def get_model_name(model_name_env: str, version: str) -> str:
    """
    根据已知 model_name 和 version 生成完整 model_name
    去掉斜杠，版本号中的点去掉
    """
    model_name_base = model_name_env.replace("/", "").lower()
    version_processed = version.replace(".", "").lower()  # 去掉点
    full_model_name = f"{model_name_base}-{version_processed}"
    return full_model_name

def get_served_model_name(model_name_env: str, version: str) -> str:
    """
    根据已知 model_name 和 version 生成 served_model_name
    去掉版本号末尾 patch
    """
    full_model_name = get_model_name(model_name_env, version)
    served_model_name = '-'.join(full_model_name.split('-')[:-1])
    return served_model_name


def get_current_namespace():
    namespace_file = "/var/run/secrets/kubernetes.io/serviceaccount/namespace"
    try:
        with open(namespace_file, 'r') as f:
            return f.read().strip()
    except Exception:
        raise Exception("Namespace file not found. Are you running in a Kubernetes Pod?")

In [4]:
from jinja2 import Template
from kubernetes import client
import yaml

api_server = "https://kubernetes.default.svc"
token = os.environ.get('OPENSHIFT_API_TOKEN')

configuration = client.Configuration()
configuration.host = api_server
configuration.api_key = {"authorization": f"Bearer {token}"}
configuration.verify_ssl = False
api_client = client.ApiClient(configuration)
api_instance = client.CustomObjectsApi(api_client)

context = {
    "model_name": get_model_name(model_name, version), #"meta-llamallama-32-1b-instruct-tuned-4",
    "served_model_name": get_served_model_name(model_name, version), #"meta-llamallama-32-1b-instruct-tuned",
    "storage_key": "models",
    "model_path": model_path, # "meta-llama/Llama-3.2-1B-Instruct-tune"
    "namespace": get_current_namespace() #"ai-ops"
}

In [5]:
context

{'model_name': 'meta-llamallama-32-1b-instruct-tuned-1',
 'served_model_name': 'meta-llamallama-32-1b-instruct-tuned',
 'storage_key': 'models',
 'model_path': 'meta-llama/Llama-3.2-1B-Instruct-tuned',
 'namespace': 'ai-ops'}

In [6]:
def resource_exists(group, version, plural, name, namespace):
    try:
        api_instance.get_namespaced_custom_object(
            group=group,
            version=version,
            namespace=namespace,
            plural=plural,
            name=name,
        )
        return True
    except client.exceptions.ApiException as e:
        if e.status == 404:
            return False
        else:
            raise

In [7]:
def apply_yaml(name, group, version, plural, yaml_content, namespace):
    resource_definition = yaml.safe_load(yaml_content)
    if resource_exists(group=group, version=version, plural=plural, name=name, namespace=namespace):
        api_instance.patch_namespaced_custom_object(
            group=group,
            version=version,
            plural=plural,
            namespace=namespace,
            name=name,
            body=resource_definition
        )
        print(f"[PATCHED] {plural}/{name} in namespace {namespace}")
    else:
        api_instance.create_namespaced_custom_object(
            group=group,
            version=version,
            plural=plural,
            namespace=namespace,
            body=resource_definition
        )
        print(f"[CREATED] {plural}/{name} in namespace {namespace}")

In [12]:
isvc_yaml = """
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  annotations:
    openshift.io/display-name: {{ model_name }}
    serving.kserve.io/deploymentMode: RawDeployment
  name: {{ model_name }}
  labels:
    modelregistry.opendatahub.io/name: model-registry
    networking.kserve.io/visibility: exposed
    opendatahub.io/dashboard: 'true'
spec:
  predictor:
    automountServiceAccountToken: false
    maxReplicas: 1
    minReplicas: 1
    model:
      modelFormat:
        name: vLLM
      name: ''
      resources:
        limits:
          cpu: '16'
          memory: 16Gi
          nvidia.com/gpu: '1'
        requests:
          cpu: '16'
          memory: 16Gi
          nvidia.com/gpu: '1'
      runtime: {{ model_name }}
      storage:
        key: {{ storage_key }}
        path: {{ model_path }}
"""

In [9]:
sr_yaml = """
apiVersion: serving.kserve.io/v1alpha1
kind: ServingRuntime
metadata:
  annotations:
    opendatahub.io/accelerator-name: ''
    opendatahub.io/apiProtocol: REST
    opendatahub.io/recommended-accelerators: '["nvidia.com/gpu"]'
    opendatahub.io/runtime-version: v0.10.1.1
    opendatahub.io/serving-runtime-scope: global
    opendatahub.io/template-display-name: vLLM NVIDIA GPU ServingRuntime for KServe
    opendatahub.io/template-name: vllm-cuda-runtime
    openshift.io/display-name: {{ model_name }}
  name: {{ model_name }}
  labels:
    opendatahub.io/dashboard: 'true'
spec:
  annotations:
    prometheus.io/path: /metrics
    prometheus.io/port: '8080'
  containers:
    - args:
        - '--port=8080'
        - '--model=/mnt/models'
        - '--served-model-name={{ served_model_name }}'
      command:
        - python
        - '-m'
        - vllm.entrypoints.openai.api_server
      env:
        - name: HF_HOME
          value: /tmp/hf_home
      image: 'registry.redhat.io/rhoai/odh-vllm-cuda-rhel9@sha256:fb84fbf103bf450ef5b060fc5f21a9cf16b166dba207a3c50aa91bccd919d604'
      name: kserve-container
      ports:
        - containerPort: 8080
          protocol: TCP
      volumeMounts:
        - mountPath: /dev/shm
          name: shm
  multiModel: false
  supportedModelFormats:
    - autoSelect: true
      name: vLLM
  volumes:
    - emptyDir:
        medium: Memory
        sizeLimit: 2Gi
      name: shm
"""

In [10]:
apply_yaml(
    name=context['model_name'],
    namespace=context['namespace'],
    group='serving.kserve.io',
    version='v1alpha1',
    plural='servingruntimes',
    yaml_content=Template(sr_yaml).render(context)
)

[CREATED] servingruntimes/meta-llamallama-32-1b-instruct-tuned-1 in namespace ai-ops


/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'kubernetes.default.svc'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'kubernetes.default.svc'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [13]:
apply_yaml(
    name=context['model_name'],
    namespace=context['namespace'],
    group='serving.kserve.io',
    version='v1beta1',
    plural='inferenceservices',
    yaml_content=Template(isvc_yaml).render(context)
)

[CREATED] inferenceservices/meta-llamallama-32-1b-instruct-tuned-1 in namespace ai-ops


/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'kubernetes.default.svc'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'kubernetes.default.svc'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
